# 03 — H&M candidate union + reranking (v2)

Reranker học cách sắp xếp lại candidate từ tower, bán chạy tuần gần nhất, mua lại, hàng mới về, content KNN.

**Vì sao v2 (số liệu từ dữ liệu `hm_v1`):** mỗi tuần chỉ ~4–5 nghìn / 105 nghìn item thực sự được bán, và ~68% item đích của tuần test đã được bán trong 7 ngày trước đó. Baseline "bán chạy tuần trước" vì vậy rất mạnh trong H&M, trong khi v1 chỉ có popularity *toàn thời gian* (top-12 của nó chỉ trúng 0,7% sự kiện test) và không có tín hiệu thời gian nào.

| Thay đổi | Lý do |
|---|---|
| Nguồn candidate mới: `recent` (bán chạy 7 ngày trước), `new_arrivals` (item xuất hiện lần đầu trong 28 ngày gần nhất, xếp theo độ giống content với lịch sử user), **`cold`** (kênh CLIP riêng cho item chưa từng bán: 0,5·z(giống lịch sử user) + z(giống hàng mới ra 56 ngày gần nhất)) | Bắt item mới/đang hot mà tower chưa biết; kênh `cold` giải bài toán cold thật sự (xem docs/logs) |
| 24 đặc trưng (v1: 7): thêm `launch_prior`, `from_cold`, độ phổ biến gần đây, tuổi item, `is_cold`, số lần user đã mua, affinity theo loại/phòng ban/màu, tỉ lệ giá so với giá user hay mua, hạng thật của tower trong catalog | Tín hiệu H&M chuẩn mà v1 thiếu |
| LightGBM LambdaRank (fallback MLP) | Tabular + nhiều đặc trưng thô; đặc trưng tính *tại thời điểm dự đoán* (cutoff từng cửa sổ) nên không rò rỉ |
| Test dùng tower **refit** (đã thấy selection+valid) và ranker refit trên selection+valid với số vòng cố định | Item mới trong 2 tuần cuối không còn cold |
| **Vị trí dành cho kênh cold** (`HM_COLD_EVERY=0,10,5`): chèn 1 item của kênh CLIP sau mỗi N vị trí của reranker | Ranker không học được cách đẩy item chưa từng bán lên top (nhãn quá hiếm); chèn vị trí cố định là cách đơn giản, đánh đổi độ chính xác tổng lấy độ phủ cold (xem `reranker+cold_slot/*` trong báo cáo) |
| Đánh giá **sampled 1+99** (uniform + popularity) cho reranker, cùng tập negative với tower-only / popularity / recent_popularity / random | Giống giao thức Amazon; luôn đọc cạnh baseline |
| Bootstrap ghép cặp theo user cho chênh lệch với baseline `recent_popularity` và với reranker kiểu v1 | Chỉ coi là cải thiện khi CI loại trừ 0 |

Tắt từng phần bằng biến môi trường (`HM_FEATURES=v1_like`, `HM_TOWER_FOR_TEST=base`, `HM_RANKER=mlp`, `HM_RANKER_REFIT=0`).

## 1. Cấu hình và dữ liệu

In [ ]:
from datetime import timedelta
from pathlib import Path
import json
import os
import random
import time

import numpy as np
import polars as pl
import torch
from torch import nn

In [ ]:
PROFILE = os.getenv("HM_PROFILE", "quick").lower()
assert PROFILE in {"quick", "full"}

DATA_DIR = Path(os.getenv("HM_DATA_DIR", os.getenv("HM_WORK_DIR", "/kaggle/input/datasets/hoho0111/hm-dataset-filled")))
RETRIEVAL_DIR = Path(os.getenv("HM_RETRIEVAL_DIR", "/kaggle/working/hm_retrieval"))
OUTPUT_DIR = Path(os.getenv("HM_RERANK_DIR", "/kaggle/working/hm_reranking"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANKER = os.getenv("HM_RANKER", "lgbm").lower()                       # lgbm | mlp
TOWER_FOR_TEST = os.getenv("HM_TOWER_FOR_TEST", "refit").lower()       # refit | base
RANKER_REFIT = os.getenv("HM_RANKER_REFIT", "1") == "1"
FEATURE_SET = os.getenv("HM_FEATURES", "v2").lower()                  # v2 | v1_like
SAMPLED_NEG = int(os.getenv("HM_SAMPLED_NEG", "99"))
DEBUG_USERS = int(os.getenv("HM_DEBUG_USERS", "0"))                   # chỉ để smoke-test nhanh
assert RANKER in {"lgbm", "mlp"} and TOWER_FOR_TEST in {"refit", "base"} and FEATURE_SET in {"v2", "v1_like"}
if RANKER == "lgbm":
    try:
        import lightgbm  # noqa: F401
    except ImportError:
        print("Không có lightgbm -> dùng MLP ranker (pip install lightgbm để dùng LambdaRank).")
        RANKER = "mlp"

cfg = json.loads((DATA_DIR / "run_config.json").read_text())
SEED = cfg["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Profile: {PROFILE} | Device: {device} | ranker={RANKER} features={FEATURE_SET} tower_for_test={TOWER_FOR_TEST} ranker_refit={RANKER_REFIT}")

In [ ]:
def find_artifact(name):
    for path in (DATA_DIR / name, DATA_DIR / "embedding" / "hm" / name, DATA_DIR / "embedding" / name):
        if path.is_file():
            return path
    raise FileNotFoundError(f"Không thấy {name} trong {DATA_DIR}")

items = pl.read_parquet(DATA_DIR / "items.parquet")
item_ids = items["item_id"].to_list()
N_ITEMS = len(item_ids)
vocab = {item_id: idx + 1 for idx, item_id in enumerate(item_ids)}

text_raw = np.load(find_artifact("text_embeddings.npy")).astype("float32")
image_raw = np.load(find_artifact("image_embeddings.npy")).astype("float32")
text = np.vstack([np.zeros((1, text_raw.shape[1]), dtype="float32"), text_raw])
image = np.vstack([np.zeros((1, image_raw.shape[1]), dtype="float32"), image_raw])

def load_events(name):
    data = pl.read_parquet(DATA_DIR / f"{name}.parquet")
    data = data.with_columns(pl.col("article_id").replace_strict(vocab, default=0).alias("idx")).filter(pl.col("idx") > 0)
    return data.sort(["customer_id", "t_dat"])

def to_history(events):
    grouped = events.group_by("customer_id", maintain_order=True).agg(pl.col("idx"))
    return dict(zip(grouped["customer_id"].to_list(), grouped["idx"].to_list()))

ev = {name: load_events(name) for name in ("train", "rerank_train", "valid", "test")}
train, selection, valid, test = (to_history(ev[n]) for n in ("train", "rerank_train", "valid", "test"))
print(f"Catalog: {N_ITEMS:,} | users train/selection/valid/test: {len(train):,}/{len(selection):,}/{len(valid):,}/{len(test):,}")

## 2. Tower đã đóng băng

`best_retrieval.pt` (train đến hết cửa sổ cũ) dùng cho cửa sổ selection và valid. `best_retrieval_refit.pt` (đã thấy selection+valid) dùng cho test nếu có và `HM_TOWER_FOR_TEST=refit`. Điểm tower được chuẩn hoá theo từng user (z-score trên toàn catalog + hạng thật) nên lệch thang đo giữa hai tower không làm hỏng ranker.

In [ ]:
MAXLEN = 30
D_MODEL = 128
base_ckpt = torch.load(RETRIEVAL_DIR / "best_retrieval.pt", map_location="cpu", weights_only=False)
MODALITY = base_ckpt.get("modality", "multimodal")
# Luật phục vụ chọn ở notebook 02 (chỉ ảnh hưởng nguồn candidate `tower`; đặc trưng vẫn dùng điểm tower thô).
SERVING = {"active_days": 0, "cold_bonus": 0.0, "pop_weight": 0.0, **(base_ckpt.get("serving") or {})}
print("serving rule từ tower:", SERVING)

class Tower(nn.Module):
    def __init__(self):
        super().__init__()
        self.id = nn.Embedding(N_ITEMS + 1, D_MODEL, padding_idx=0)
        self.text_proj = nn.Linear(text.shape[1], D_MODEL, bias=False)
        self.image_proj = nn.Linear(image.shape[1], D_MODEL, bias=False)
        self.image_gate = nn.Parameter(torch.tensor(0.0))
        self.id_gate = nn.Parameter(torch.tensor(-2.0))
        self.register_buffer("text", torch.tensor(text))
        self.register_buffer("image", torch.tensor(image))
        self.register_buffer("id_keep", torch.ones(N_ITEMS + 1, 1))
        self.register_buffer("degree", torch.zeros(N_ITEMS + 1))

    def item_table(self):
        id_vectors = torch.sigmoid(self.id_gate) * self.id.weight * self.id_keep
        if MODALITY == "id":
            vectors = id_vectors
        elif MODALITY == "text":
            vectors = self.text_proj(self.text)
        elif MODALITY == "image":
            vectors = self.image_proj(self.image)
        else:
            vectors = id_vectors + self.text_proj(self.text) + torch.sigmoid(self.image_gate) * self.image_proj(self.image)
        return nn.functional.normalize(vectors, dim=1)

    def query(self, history, table):
        mask = history.ne(0)
        weights = torch.arange(1, history.shape[1] + 1, device=history.device)[None] * mask
        query = (table[history] * weights[:, :, None]).sum(1) / weights.sum(1, keepdim=True).clamp_min(1)
        return nn.functional.normalize(query, dim=1)

def load_tower(path):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = Tower().to(device)
    model.load_state_dict(ckpt["state"])
    model.eval()
    degree = np.asarray(ckpt["degree"], dtype="float64")
    return {"model": model, "table": model.item_table().detach(), "degree": degree, "name": path.stem, "cold_t": torch.tensor(degree == 0, device=device)}

towers = {"base": load_tower(RETRIEVAL_DIR / "best_retrieval.pt")}
refit_path = RETRIEVAL_DIR / "best_retrieval_refit.pt"
if TOWER_FOR_TEST == "refit" and refit_path.is_file():
    towers["refit"] = load_tower(refit_path)
else:
    if TOWER_FOR_TEST == "refit":
        print("Không thấy best_retrieval_refit.pt -> test dùng tower base.")
    towers["refit"] = towers["base"]
print({k: (v["name"], int((v["degree"] > 0).sum())) for k, v in towers.items()})

## 3. Thông tin item theo thời điểm (không rò rỉ)

Với mỗi cửa sổ đích, mọi thống kê item (độ phổ biến, bán chạy 7 ngày, tuổi, giá) chỉ dùng sự kiện **trước ngày bắt đầu cửa sổ**. Cutoff: selection = ngày đầu cửa sổ `rerank_train`, valid = ngày đầu `valid`, test = ngày đầu `test`.

Nếu có `item_daily_counts.parquet` (do notebook 00 xuất từ **toàn bộ** giao dịch của mọi khách), thống kê item lấy từ đó: đây là tín hiệu cấp item, không cá nhân hoá và chỉ dùng ngày < cutoff nên không rò rỉ nhãn; production luôn có sẵn. Nếu không có, rơi về mẫu 50k khách (nhiễu hơn ~27 lần).

In [ ]:
all_events = pl.concat([ev[n].select("idx", "t_dat", "price") for n in ("train", "rerank_train", "valid", "test")])
last_day = int(ev["test"]["t_dat"].cast(pl.Int32).max())
CUTOFF = {"selection": last_day - 20, "valid": last_day - 13, "test": last_day - 6}   # 3 cửa sổ 7 ngày liên tiếp

# Thống kê item (độ phổ biến, bán chạy, tuổi, giá): ưu tiên bảng đếm theo ngày của TOÀN BỘ khách hàng (notebook 00 xuất ra
# item_daily_counts.parquet) vì mẫu 50k khách chỉ thấy ~4% giao dịch -> đếm nhiễu và bỏ sót item đang bán. Chỉ dùng sự kiện < cutoff.
GLOBAL_STATS = Path(os.getenv("HM_GLOBAL_STATS", str(DATA_DIR / "item_daily_counts.parquet")))
if GLOBAL_STATS.is_file():
    counts = pl.read_parquet(GLOBAL_STATS).with_columns(pl.col("article_id").replace_strict(vocab, default=0).alias("idx")).filter(pl.col("idx") > 0)
    st_idx, st_day = counts["idx"].to_numpy(), counts["t_dat"].cast(pl.Int32).to_numpy()
    st_n, st_psum = counts["n"].to_numpy().astype("float64"), counts["price_sum"].to_numpy().astype("float64")
    STATS_SOURCE = f"global ({int(st_n.sum()):,} giao dịch)"
else:
    st_idx, st_day = all_events["idx"].to_numpy(), all_events["t_dat"].cast(pl.Int32).to_numpy()
    st_n, st_psum = np.ones(len(st_idx)), all_events["price"].cast(pl.Float64).to_numpy()
    STATS_SOURCE = f"mẫu 50k khách ({len(st_idx):,} giao dịch) - chạy notebook 00 v2 để có item_daily_counts.parquet"
print("item stats từ:", STATS_SOURCE)
first_seen = np.full(N_ITEMS + 1, 10 ** 9, dtype="int64")
np.minimum.at(first_seen, st_idx, st_day)

def l2_normalize(values):
    return values / np.linalg.norm(values, axis=1, keepdims=True).clip(1e-8)

content_np = np.vstack([np.zeros((1, text_raw.shape[1]), dtype="float32"), l2_normalize(l2_normalize(text_raw) + l2_normalize(image_raw))])
content_t = torch.tensor(content_np, device=device)

COLD_EVERY = [int(x) for x in os.getenv("HM_COLD_EVERY", "0,10,5").split(",")]   # chèn 1 item kênh cold sau mỗi N vị trí (0 = không chèn)
COLD_K = int(os.getenv("HM_COLD_K", "100"))      # số candidate của kênh cold CLIP (0 = tắt)
PRIOR_DAYS, PRIOR_TOPK = 56, 10

def launch_prior(cutoff):
    """Độ giống nội dung (CLIP) của từng item với các item MỚI RA trong PRIOR_DAYS ngày trước cutoff (trung bình top-k cosine).
    Item mới ra thường cùng bộ sưu tập/mùa với hàng mới ra gần đây -> prior "có khả năng được bán tuần tới" cho item chưa từng bán."""
    new_ids = np.where((first_seen < cutoff) & (first_seen >= cutoff - PRIOR_DAYS))[0]
    prior = np.zeros(N_ITEMS + 1, dtype="float32")
    if len(new_ids) == 0:
        return prior
    new_vecs = content_t[torch.from_numpy(new_ids).to(device)]
    for start in range(1, N_ITEMS + 1, 8192):
        block = content_t[start:start + 8192] @ new_vecs.T
        prior[start:start + block.shape[0]] = block.topk(min(PRIOR_TOPK, len(new_ids)), dim=1).values.mean(1).cpu().numpy()
    return prior

def item_stats(cutoff):
    before = st_day < cutoff
    pop_all = np.bincount(st_idx[before], weights=st_n[before], minlength=N_ITEMS + 1)
    recent = before & (st_day >= cutoff - 7)
    pop_recent = np.bincount(st_idx[recent], weights=st_n[recent], minlength=N_ITEMS + 1)
    price_sum = np.bincount(st_idx[before], weights=st_psum[before], minlength=N_ITEMS + 1)
    price = np.where(pop_all > 0, price_sum / np.maximum(pop_all, 1), np.nan)
    price = np.where(np.isnan(price), np.nanmedian(price), price).clip(1e-4)
    age = np.where(first_seen < cutoff, cutoff - first_seen, -1)
    inactive = None
    if SERVING["active_days"]:
        active = np.bincount(st_idx[(st_day < cutoff) & (st_day >= cutoff - SERVING["active_days"])], minlength=N_ITEMS + 1) > 0
        inactive = torch.tensor(~active, device=device)
    cold_pool = np.where(first_seen[1:] >= cutoff)[0] + 1          # item chưa có giao dịch nào trước cutoff (cold thật sự)
    prior = launch_prior(cutoff)
    pool_mean, pool_std = (prior[cold_pool].mean(), prior[cold_pool].std() + 1e-6) if len(cold_pool) else (0.0, 1.0)
    prior_z = (prior - pool_mean) / pool_std
    cold_pool_t = torch.from_numpy(cold_pool).to(device)
    return {"cold_pool_t": cold_pool_t, "cold_pool_vecs": content_t[cold_pool_t], "cold_prior_z_t": torch.tensor(prior_z[cold_pool], device=device),
            "prior_z": prior_z, "inactive_t": inactive, "log_recent_t": torch.tensor(np.log1p(pop_recent), dtype=torch.float32, device=device), "pop_all": pop_all, "pop_recent": pop_recent, "price": price, "age": age.astype("float64"), "seen": (first_seen < cutoff).astype("float64"),
            "recent_ranked": [int(i) for i in np.argsort(-pop_recent, kind="stable") if i != 0],
            "popular_ranked": [int(i) for i in np.argsort(-pop_all, kind="stable") if i != 0],
            "new_pool": np.where((first_seen < cutoff) & (first_seen >= cutoff - 28))[0]}

STATS = {w: item_stats(c) for w, c in CUTOFF.items()}
print({w: {"recent_items": int((s["pop_recent"] > 0).sum()), "new_arrivals_pool": len(s["new_pool"]), "cold_pool": len(s["cold_pool_t"])} for w, s in STATS.items()})

def category_codes(column):
    codes = items[column].fill_null("").cast(pl.Categorical).to_physical().to_numpy().astype("int64") + 1
    return np.concatenate([[0], codes])
ATTR = {"type": category_codes("product_type_name"), "dept": category_codes("department_name"), "colour": category_codes("colour_group_name")}

## 4. Candidate union và đặc trưng

Nguồn: `tower` (500) · `recent` (200) · `repeat` (100) · `new_arrivals` (100) · `content` (200) · `popular` (100). Ground truth không bao giờ được chèn vào candidate. `featurize()` dùng được cho *bất kỳ* item (kể cả negative của giao thức 1+99), nên cả hai kiểu đánh giá dùng chung một định nghĩa đặc trưng.

In [ ]:
TOWER_K, RECENT_K, REPEAT_K, NEW_K, CONTENT_K, POPULAR_K = 500, 200, 100, 100, 200, 100
SOURCES = ["tower", "recent", "repeat", "new", "content", "popular", "cold"]
FEATURES_V2 = ["tower_z", "tower_logrank", "log_pop_all", "log_pop_recent", "recent_share", "content_anchor", "content_mean",
               "from_tower", "from_recent", "from_repeat", "from_new", "from_content", "from_popular",
               "is_cold", "repeat_count", "age_days", "seen_before", "type_aff", "dept_aff", "colour_aff", "same_type_last", "log_price_ratio", "from_cold", "launch_prior"]
FEATURES_V1_LIKE = ["tower_z", "log_pop_all", "content_anchor", "from_tower", "from_popular", "from_repeat", "from_content"]
FEATURES = FEATURES_V2 if FEATURE_SET == "v2" else FEATURES_V1_LIKE
COL = {name: i for i, name in enumerate(FEATURES_V2)}
FEATURE_COLUMNS = [COL[name] for name in FEATURES]

def unique_recent(history, limit):
    return list(dict.fromkeys(reversed(history)))[:limit]

@torch.no_grad()
def batch_state(tower, context, users, stats):
    """Điểm tower / content cho cả batch user trên GPU; chỉ chuyển sang CPU phần cần thiết."""
    histories = [context.get(u, [])[-MAXLEN:] for u in users]
    X = torch.tensor([[0] * (MAXLEN - len(h)) + h for h in histories], device=device)
    scores = tower["model"].query(X, tower["table"]) @ tower["table"].T
    scores[:, 0] = -torch.inf
    anchor3 = nn.functional.normalize(content_t[X[:, -3:]].sum(1), dim=1)
    anchor_all = nn.functional.normalize(content_t[X].sum(1), dim=1)
    anchor10 = nn.functional.normalize(content_t[X[:, -10:]].sum(1), dim=1)
    cold_top = np.zeros((len(users), 0), dtype="int64")
    if COLD_K and len(stats["cold_pool_t"]):
        # Kênh cold: chỉ xếp hạng item chưa từng bán, điểm = 0.5*z(độ giống lịch sử user) + z(prior hàng mới ra)
        usim = anchor10 @ stats["cold_pool_vecs"].T
        usim_z = (usim - usim.mean(1, keepdim=True)) / usim.std(1, keepdim=True).clamp_min(1e-6)
        pick = (0.5 * usim_z + stats["cold_prior_z_t"][None]).topk(min(COLD_K, usim.shape[1]), dim=1).indices
        cold_top = stats["cold_pool_t"][pick].cpu().numpy()
    content_scores = anchor3 @ content_t.T
    content_scores[:, 0] = -torch.inf
    serve = scores.clone()
    if SERVING["pop_weight"]:
        serve += SERVING["pop_weight"] * stats["log_recent_t"][None]
    if SERVING["cold_bonus"]:
        serve[:, tower["cold_t"]] += SERVING["cold_bonus"]
    if stats["inactive_t"] is not None:
        serve[:, stats["inactive_t"]] = -torch.inf
    serve[:, 0] = -torch.inf
    return {"scores": scores, "sorted": scores.sort(dim=1).values, "mu": scores[:, 1:].mean(1), "sd": scores[:, 1:].std(1).clamp_min(1e-6),
            "tower_top": serve.topk(TOWER_K, dim=1).indices.cpu().numpy(), "content_top": content_scores.topk(CONTENT_K, dim=1).indices.cpu().numpy(),
            "anchor3": anchor3, "anchor_all": anchor_all, "cold_top": cold_top}

def build_sources(b, i, hist_full, stats):
    new_pool = stats["new_pool"]
    new_top = []
    if len(new_pool):
        sim = content_t[torch.from_numpy(new_pool).to(device)] @ b["anchor3"][i]
        new_top = new_pool[sim.topk(min(NEW_K, len(new_pool))).indices.cpu().numpy()].tolist()
    return {"tower": b["tower_top"][i].tolist(), "recent": stats["recent_ranked"][:RECENT_K], "repeat": unique_recent(hist_full, REPEAT_K),
            "new": new_top, "content": b["content_top"][i].tolist(), "popular": stats["popular_ranked"][:POPULAR_K], "cold": b["cold_top"][i].tolist()}

def featurize(b, i, items_arr, hist_full, sources, stats, tower):
    """Ma trận đặc trưng (len(items_arr), 24) cho một user + điểm tower thô."""
    items_arr = np.asarray(items_arr, dtype="int64")
    t = torch.from_numpy(items_arr).to(device)
    raw = b["scores"][i, t]
    z = ((raw - b["mu"][i]) / b["sd"][i]).cpu().numpy()
    higher = (N_ITEMS + 1 - torch.searchsorted(b["sorted"][i], raw, right=True)).cpu().numpy()  # số item có điểm cao hơn
    content_anchor = (content_t[t] @ b["anchor3"][i]).cpu().numpy()
    content_mean = (content_t[t] @ b["anchor_all"][i]).cpu().numpy()
    flags = [np.isin(items_arr, np.asarray(sources[name], dtype="int64")).astype("float32") for name in SOURCES]

    hist = np.asarray(hist_full, dtype="int64")
    hist_counts = np.bincount(hist, minlength=N_ITEMS + 1) if len(hist) else np.zeros(N_ITEMS + 1, dtype="int64")
    recent_hist = hist[-MAXLEN:]
    affinity = {}
    for name, codes in ATTR.items():
        share = np.bincount(codes[recent_hist], minlength=codes.max() + 1) / max(len(recent_hist), 1) if len(recent_hist) else np.zeros(codes.max() + 1)
        affinity[name] = share[codes[items_arr]]
    same_type_last = (ATTR["type"][items_arr] == ATTR["type"][hist[-1]]).astype("float32") if len(hist) else np.zeros(len(items_arr), dtype="float32")
    user_price = stats["price"][recent_hist].mean() if len(recent_hist) else np.median(stats["price"])

    pop_all, pop_recent = stats["pop_all"][items_arr], stats["pop_recent"][items_arr]
    columns = {
        "tower_z": z, "tower_logrank": np.log1p(higher), "log_pop_all": np.log1p(pop_all), "log_pop_recent": np.log1p(pop_recent),
        "recent_share": pop_recent / (pop_all + 1.0), "content_anchor": content_anchor, "content_mean": content_mean,
        "from_tower": flags[0], "from_recent": flags[1], "from_repeat": flags[2], "from_new": flags[3], "from_content": flags[4], "from_popular": flags[5],
        "from_cold": flags[6], "launch_prior": stats["prior_z"][items_arr],
        "is_cold": (tower["degree"][items_arr] == 0).astype("float32"), "repeat_count": np.log1p(hist_counts[items_arr]),
        "age_days": np.clip(stats["age"][items_arr], -1, 800) / 100.0, "seen_before": stats["seen"][items_arr],
        "type_aff": affinity["type"], "dept_aff": affinity["dept"], "colour_aff": affinity["colour"], "same_type_last": same_type_last,
        "log_price_ratio": np.log(stats["price"][items_arr] / user_price),
    }
    matrix = np.column_stack([columns[name] for name in FEATURES_V2]).astype("float32")
    return matrix, raw.cpu().numpy()

In [ ]:
def draw_negatives(rng, n_rows, n_neg, forbidden, cdf=None):
    out = np.empty((n_rows, n_neg), dtype=np.int64)
    forbidden = np.fromiter(forbidden, dtype=np.int64)
    for row in range(n_rows):
        got = []
        while len(got) < n_neg:
            need = n_neg - len(got)
            draws = rng.integers(1, N_ITEMS + 1, size=need * 2) if cdf is None else np.clip(np.searchsorted(cdf, rng.random(need * 2)) + 1, 1, N_ITEMS)
            draws = draws[~np.isin(draws, forbidden)]
            got = list(dict.fromkeys(got + draws.tolist()))[:n_neg]
        out[row] = got
    return out

def build_window(window, context, targets, tower, sampled=False, batch_size=32):
    """Groups (union + đặc trưng) cho một cửa sổ; sampled=True thêm bộ 1+99 (uniform, popularity) dùng chung cho mọi bộ chấm."""
    stats = STATS[window]
    users = list(targets)
    if DEBUG_USERS:
        users = users[:DEBUG_USERS]
    weights = np.maximum(tower["degree"][1:], 1e-6)
    cdf_pop = np.cumsum(weights / weights.sum()); cdf_pop[-1] = 1.0
    rng = np.random.default_rng(SEED)
    groups = []
    pair_cache = {s: {"cands": [], "X": [], "raw": [], "user": []} for s in ("uniform", "popularity")} if sampled else None
    for start in range(0, len(users), batch_size):
        chunk = users[start:start + batch_size]
        b = batch_state(tower, context, chunk, stats)
        for i, user in enumerate(chunk):
            hist_full = context.get(user, [])
            sources = build_sources(b, i, hist_full, stats)
            union = list(dict.fromkeys(item for name in SOURCES for item in sources[name] if item != 0))
            X, raw = featurize(b, i, union, hist_full, sources, stats, tower)
            truth = set(targets[user])
            groups.append({"user": user, "cold_list": sources["cold"], "history": set(hist_full), "truth": truth, "items": np.asarray(union), "x": X, "raw": raw})
            if sampled and truth:
                positives = sorted(truth)
                forbidden = set(hist_full) | truth | {0}
                for strategy, cdf in (("uniform", None), ("popularity", cdf_pop)):
                    negs = draw_negatives(rng, len(positives), SAMPLED_NEG, forbidden, cdf)
                    cands = np.concatenate([np.asarray(positives)[:, None], negs], axis=1)
                    Xs, raws = featurize(b, i, cands.reshape(-1), hist_full, sources, stats, tower)
                    cache = pair_cache[strategy]
                    cache["cands"].append(cands); cache["X"].append(Xs.reshape(len(positives), 1 + SAMPLED_NEG, -1)); cache["raw"].append(raws.reshape(len(positives), -1))
                    cache["user"].extend([len(groups) - 1] * len(positives))
    if sampled:
        for cache in pair_cache.values():
            for key in ("cands", "X", "raw"):
                cache[key] = np.concatenate(cache[key])
            cache["user"] = np.asarray(cache["user"])
    return groups, pair_cache

def candidate_metrics(groups, target_filter=None):
    recalls, hits, covered = [], [], set()
    for group in groups:
        truth = {item for item in group["truth"] if target_filter is None or target_filter(group, item)}
        if not truth:
            continue
        found = truth.intersection(group["items"].tolist())
        recalls.append(len(found) / len(truth)); hits.append(float(bool(found))); covered.update(group["items"].tolist())
    return {"evaluated_users": len(recalls), "CandidateRecall": float(np.mean(recalls) if recalls else 0), "CandidateHitRate": float(np.mean(hits) if hits else 0),
            "CandidateSize": float(np.mean([len(g["items"]) for g in groups])), "CatalogCoverage": len(covered) / N_ITEMS}

In [ ]:
t0 = time.time()
train_groups, _ = build_window("selection", train, selection, towers["base"])
print(f"selection groups: {len(train_groups)} ({time.time() - t0:.0f}s)")
valid_context = {u: train.get(u, []) + selection.get(u, []) for u in valid}
valid_groups, valid_sampled = build_window("valid", valid_context, valid, towers["base"], sampled=True)
print(f"valid groups: {len(valid_groups)} ({time.time() - t0:.0f}s)")
test_context = {u: train.get(u, []) + selection.get(u, []) + valid.get(u, []) for u in test}
test_tower = towers["refit"]
test_groups, test_sampled = build_window("test", test_context, test, test_tower, sampled=True)
print(f"test groups: {len(test_groups)} ({time.time() - t0:.0f}s)")
print({"selection": candidate_metrics(train_groups), "valid": candidate_metrics(valid_groups), "test": candidate_metrics(test_groups)})

## 5. Reranker

Nhãn = item thực sự được mua trong cửa sổ đích. Train trên các group của cửa sổ `selection` (tower chưa thấy nhãn này), dừng sớm theo `valid`. Group không có positive trong union bị bỏ khi train.

In [ ]:
def stack_groups(groups, columns):
    kept = [g for g in groups if g["truth"] & set(g["items"].tolist())]
    X = np.concatenate([g["x"][:, columns] for g in kept]).astype("float32")
    y = np.concatenate([np.isin(g["items"], list(g["truth"])).astype("float32") for g in kept])
    return X, y, [len(g["items"]) for g in kept]

class MLPRanker(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, 128), nn.GELU(), nn.Dropout(0.1), nn.Linear(128, 64), nn.GELU(), nn.Linear(64, 1))
    def forward(self, features):
        return self.net(features).squeeze(-1)

class Scorer:
    """Giao diện chung: predict(X[n, 24]) -> điểm (n,)."""
    def __init__(self, kind, model, columns, mean=None, std=None):
        self.kind, self.model, self.columns, self.mean, self.std = kind, model, columns, mean, std
    def predict(self, X):
        X = X[:, self.columns]
        if self.kind == "lgbm":
            return self.model.predict(X, num_iteration=self.model.best_iteration or None)
        with torch.no_grad():
            self.model.eval()
            return self.model(torch.tensor((X - self.mean) / self.std, dtype=torch.float32, device=device)).cpu().numpy()

LGB_PARAMS = dict(objective="lambdarank", metric="ndcg", ndcg_eval_at=[12], learning_rate=0.05, num_leaves=63, min_data_in_leaf=50,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=SEED)

def fit_ranker(fit_groups, columns, valid_groups=None, rounds=None):
    X, y, sizes = stack_groups(fit_groups, columns)
    print(f"  fit on {len(sizes)} groups / {len(y):,} rows / {int(y.sum())} positives / {len(columns)} features")
    if RANKER == "lgbm":
        import lightgbm as lgb
        train_set = lgb.Dataset(X, label=y, group=sizes)
        if valid_groups is not None:
            Xv, yv, sv = stack_groups(valid_groups, columns)
            booster = lgb.train(LGB_PARAMS, train_set, num_boost_round=1500, valid_sets=[lgb.Dataset(Xv, label=yv, group=sv)],
                                callbacks=[lgb.early_stopping(60, verbose=False), lgb.log_evaluation(100)])
        else:
            booster = lgb.train(LGB_PARAMS, train_set, num_boost_round=rounds)
        return Scorer("lgbm", booster, columns), (booster.best_iteration or booster.current_iteration())
    mean, std = X.mean(0), X.std(0).clip(1e-6)
    model = MLPRanker(len(columns)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    offsets = np.concatenate([[0], np.cumsum(sizes)])
    Xn = torch.tensor((X - mean) / std, dtype=torch.float32, device=device); yt = torch.tensor(y, device=device)
    epochs = rounds or int(os.getenv("HM_RERANK_EPOCHS", 6 if PROFILE == "quick" else 15))
    best, best_state, best_epoch = -1, None, epochs
    for epoch in range(1, epochs + 1):
        model.train()
        for g in np.random.permutation(len(sizes)):
            sl = slice(offsets[g], offsets[g + 1])
            labels = yt[sl]
            loss = -(torch.log_softmax(model(Xn[sl]), 0) * labels / labels.sum()).sum()
            optimizer.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5); optimizer.step()
        scorer = Scorer("mlp", model, columns, mean, std)
        if valid_groups is None:
            continue
        score = ranking_summary(valid_groups, scorer)["NDCG@12"]
        print(f"  epoch {epoch}: valid NDCG@12={score:.4f}")
        if score > best:
            best, best_epoch, best_state = score, epoch, {k: v.detach().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return Scorer("mlp", model, columns, mean, std), best_epoch

### Hàm đánh giá (full-ranking trên union + sampled 1+99)

Full-ranking giữ định nghĩa v1 (HitRate/Recall/MAP/NDCG@K trên toàn catalog — item ngoài union coi như không được gợi ý). `sampled` dùng đúng tập negative đã cache ở trên cho mọi bộ chấm nên so sánh ghép cặp được.

In [ ]:
def rank_groups(groups, scorer):
    rankings = {}
    for g in groups:
        scores = scorer.predict(g["x"])
        rankings[g["user"]] = g["items"][np.argsort(-scores, kind="stable")].tolist()
    return rankings

def interleave_cold(ranked, cold_list, every):
    """Giữ nguyên thứ tự reranker, nhưng dành vị trí every, 2*every, ... cho item của kênh cold (theo thứ tự điểm CLIP của kênh)."""
    cold_set = set(cold_list)
    main = [x for x in ranked if x not in cold_set]
    out, mi, ci = [], 0, 0
    while mi < len(main) or ci < len(cold_list):
        if ((len(out) + 1) % every == 0 and ci < len(cold_list)) or mi >= len(main):
            out.append(cold_list[ci]); ci += 1
        else:
            out.append(main[mi]); mi += 1
    return out

def per_user_metrics(rankings, targets, ks=(12, 50, 100), target_filter=None):
    top = max(ks)
    discounts = 1 / np.log2(np.arange(top) + 2)
    out = {k: {"hit": {}, "recall": {}, "ndcg": {}, "map": {}} for k in ks}
    positions = np.arange(1, top + 1)
    for user, truth in targets.items():
        relevant = {i for i in truth if target_filter is None or target_filter(user, i)}
        if not relevant or user not in rankings:
            continue
        hits = np.isin(np.asarray(rankings[user][:top]), list(relevant)).astype("float64")
        precision = np.cumsum(hits) / positions[:len(hits)]
        for k in ks:
            n_rel = len(relevant)
            out[k]["hit"][user] = float(hits[:k].any()); out[k]["recall"][user] = float(hits[:k].sum()) / n_rel
            out[k]["ndcg"][user] = float((hits[:k] * discounts[:len(hits)][:k]).sum()) / float(discounts[:min(k, n_rel)].sum())
            out[k]["map"][user] = float((precision[:k] * hits[:k]).sum()) / min(k, n_rel)
    return out

def summarize(per_user, ks=(12, 50, 100)):
    result = {"evaluated_users": len(per_user[ks[0]]["hit"])}
    for k in ks:
        for key, name in (("hit", "HitRate"), ("recall", "Recall"), ("ndcg", "NDCG"), ("map", "MAP")):
            vals = list(per_user[k][key].values())
            result[f"{name}@{k}"] = float(np.mean(vals)) if vals else 0.0
    return result

def ranking_summary(groups, scorer):
    targets = {g["user"]: g["truth"] for g in groups}
    return summarize(per_user_metrics(rank_groups(groups, scorer), targets))

def cluster_ci(values, user_index, n_users, z=1.96):
    values = np.asarray(values, dtype="float64")
    if len(values) == 0:
        return float("nan")
    sums = np.bincount(user_index, weights=values - values.mean(), minlength=n_users)
    return float(z * np.sqrt((sums ** 2).sum()) / len(values))

def sampled_report(cache, scorers, n_groups, positive_mask=None, ks=(10, 20, 50)):
    """scorers: {tên: fn(X[P,100,22], raw[P,100]) -> điểm (P,100)}; đồng hạng tính trung bình."""
    keep = np.ones(len(cache["cands"]), dtype=bool) if positive_mask is None else positive_mask
    out = {}
    for name, fn in scorers.items():
        scores = fn(cache["X"][keep], cache["raw"][keep])
        pos, neg = scores[:, :1], scores[:, 1:]
        ranks = (neg > pos).sum(1) + 0.5 * (neg == pos).sum(1)
        row = {"n_pairs": int(len(ranks)), "MRR": float((1 / (ranks + 1)).mean())}
        for k in ks:
            hit = (ranks < k).astype("float64")
            row[f"HitRate@{k}"] = float(hit.mean()); row[f"HitRate@{k}_ci95"] = cluster_ci(hit, cache["user"][keep], n_groups)
            row[f"NDCG@{k}"] = float(np.where(ranks < k, 1 / np.log2(ranks + 2), 0).mean())
        out[name] = row
    return out

FROM_COLUMNS = [COL[f"from_{name}"] for name in ("tower", "recent", "repeat", "new", "content", "popular", "cold")]

def make_scorers(ranker_scorers):
    """`<ranker>` chấm thẳng mọi item bằng cùng bộ đặc trưng (giống tower của Amazon chấm cả catalog); negative ngẫu nhiên nằm ngoài
    phân phối candidate-union lúc train nên đây là phép đo "ranker như hàm chấm điểm". `reranker_union_gated` = hệ thống thật khi item ngoài union
    không bao giờ được gợi ý: các negative gần như không vào union nên số này ≈ recall của union (HR@10 = HR@20), chỉ để tham khảo."""
    rng = np.random.default_rng(SEED)
    base = {
        "tower_only": lambda X, raw: raw,
        "popularity": lambda X, raw: X[..., COL["log_pop_all"]],
        "recent_popularity": lambda X, raw: X[..., COL["log_pop_recent"]],
        "random": lambda X, raw: rng.random(raw.shape),
    }
    def raw_score(scorer, X):
        return scorer.predict(X.reshape(-1, X.shape[-1])).reshape(X.shape[:-1])
    for name, scorer in ranker_scorers.items():
        base[name] = (lambda s: lambda X, raw: raw_score(s, X))(scorer)
    if "reranker" in ranker_scorers:
        main = ranker_scorers["reranker"]
        base["reranker_union_gated"] = lambda X, raw: np.where(X[..., FROM_COLUMNS].max(-1) > 0, raw_score(main, X), -1e9)
    return base

In [ ]:
FEATURE_VARIANTS = {"reranker": list(range(len(FEATURES_V2)))}
if FEATURE_SET == "v2":
    FEATURE_VARIANTS = {"reranker": [COL[n] for n in FEATURES_V2], "reranker_v1_features": [COL[n] for n in FEATURES_V1_LIKE]}
else:
    FEATURE_VARIANTS = {"reranker": [COL[n] for n in FEATURES_V1_LIKE]}

rankers_selection, best_rounds = {}, {}
for name, columns in FEATURE_VARIANTS.items():
    print(f"[{name}] huấn luyện trên cửa sổ selection, dừng sớm theo valid")
    rankers_selection[name], best_rounds[name] = fit_ranker(train_groups, columns, valid_groups)
    print(f"  -> best round/epoch {best_rounds[name]} | valid:", {k: round(v, 4) for k, v in ranking_summary(valid_groups, rankers_selection[name]).items() if k in ("HitRate@12", "NDCG@12", "Recall@100")})

### Refit ranker cho test

Gộp group `selection` + `valid` (đều xây bằng tower chưa thấy nhãn của chính nó) và train với đúng số vòng đã chọn. Test vẫn chưa được dùng.

In [ ]:
rankers_test = {}
for name, columns in FEATURE_VARIANTS.items():
    if RANKER_REFIT:
        rounds = max(int(best_rounds[name] * 1.1), 10) if RANKER == "lgbm" else best_rounds[name]
        print(f"[{name}] refit trên selection+valid, {rounds} vòng")
        rankers_test[name], _ = fit_ranker(train_groups + valid_groups, columns, None, rounds=rounds)
    else:
        rankers_test[name] = rankers_selection[name]

if RANKER == "lgbm":
    booster = rankers_test["reranker"].model
    gain = booster.feature_importance("gain")
    importance = sorted(zip([FEATURES_V2[c] for c in FEATURE_VARIANTS["reranker"]], gain / gain.sum()), key=lambda x: -x[1])
    print("feature gain:", [(n, round(float(g), 3)) for n, g in importance])

## 6. Kết quả

* `valid` dùng ranker train trên selection (early-stop theo valid → hơi lạc quan, chỉ để tham khảo).
* `test` dùng tower refit + ranker refit → **số báo cáo**. Cold = item chưa có tương tác trong tập fit của *chính tower dùng cho cửa sổ đó*.
* So sánh có ý nghĩa khi CI 95% ghép cặp của chênh lệch loại trừ 0.

In [ ]:
def paired_diff(a, b, n_boot=2000, seed=SEED):
    users = sorted(set(a) & set(b))
    d = np.array([a[u] - b[u] for u in users])
    rng = np.random.default_rng(seed)
    boots = rng.choice(d, size=(n_boot, len(d)), replace=True).mean(1)
    return {"diff": float(d.mean()), "ci95": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))], "users": len(users)}

def full_report(groups, ranker_scorers, tower, window, context):
    degree = tower["degree"]
    targets = {g["user"]: g["truth"] for g in groups}
    warm = lambda _, i: degree[i] > 0
    cold = lambda _, i: degree[i] == 0
    repeat = lambda u, i: i in context.get(u, [])
    explore = lambda u, i: i not in context.get(u, [])
    never_sold = lambda _, i: STATS[window]["seen"][i] == 0     # chưa có giao dịch nào trước cutoff (cold thật sự, không có lịch sử bán)
    slices = {"overall": None, "warm_items": warm, "strict_cold_items": cold, "never_sold_items": never_sold, "repeat_items": repeat, "explore_items": explore}
    recent_top = STATS[window]["recent_ranked"][:1000]
    rankings = {name: rank_groups(groups, s) for name, s in ranker_scorers.items()}
    if "reranker" in ranker_scorers:
        for every in COLD_EVERY:
            if every and COLD_K:
                rankings[f"reranker+cold_slot/{every}"] = {g["user"]: interleave_cold(r, g["cold_list"], every) for g, r in ((g, rankings["reranker"][g["user"]]) for g in groups)}
    rankings["recent_popularity"] = {g["user"]: recent_top for g in groups}
    rankings["popularity"] = {g["user"]: STATS[window]["popular_ranked"][:1000] for g in groups}
    report = {"candidate": {s: candidate_metrics(groups, (lambda g, i, f=f: f(g["user"], i)) if f else None) for s, f in slices.items()}, "full_ranking": {}, "paired_vs_recent_popularity": {}}
    per_user = {}
    for name, ranking in rankings.items():
        report["full_ranking"][name] = {}
        for slice_name, flt in slices.items():
            pu = per_user_metrics(ranking, targets, target_filter=flt)
            report["full_ranking"][name][slice_name] = summarize(pu)
            if slice_name == "overall":
                per_user[name] = pu
    for name in ranker_scorers:
        for metric in ("hit", "ndcg"):
            report["paired_vs_recent_popularity"][f"{name}/{metric}@12"] = paired_diff(per_user[name][12][metric], per_user["recent_popularity"][12][metric])
    if "reranker" in ranker_scorers and "reranker_v1_features" in ranker_scorers:
        for metric in ("hit", "ndcg"):
            report.setdefault("paired_v2_vs_v1_features", {})[f"{metric}@12"] = paired_diff(per_user["reranker"][12][metric], per_user["reranker_v1_features"][12][metric])
    return report

def evaluate_window(groups, sampled_cache, ranker_scorers, tower, window, context):
    report = full_report(groups, ranker_scorers, tower, window, context)
    degree = tower["degree"]
    scorers = make_scorers(ranker_scorers)
    report["sampled_1plus99"] = {}
    for strategy, cache in sampled_cache.items():
        pos_warm = degree[cache["cands"][:, 0]] > 0
        for slice_name, mask in (("overall", None), ("warm_items", pos_warm), ("strict_cold_items", ~pos_warm)):
            chosen = scorers if mask is None else {k: v for k, v in scorers.items() if k in ("tower_only", "reranker", "random")}
            report["sampled_1plus99"][f"{strategy}/{slice_name}"] = sampled_report(cache, chosen, len(groups), mask)
    report["cold_target_share"] = float(np.mean([degree[i] == 0 for g in groups for i in g["truth"]]))
    return report

def print_report(title, report):
    print(f"\n===== {title} | cold target share {report['cold_target_share']:.1%} =====")
    print(f"{'candidate union':<28}{'recall':>8}{'hit':>8}{'size':>7}")
    for s, m in report["candidate"].items():
        print(f"{s:<28}{m['CandidateRecall']:>8.3f}{m['CandidateHitRate']:>8.3f}{m['CandidateSize']:>7.0f}")
    print(f"\n{'full-ranking (overall)':<28}{'Hit@12':>8}{'Hit@50':>8}{'Hit@100':>9}{'Rec@100':>9}{'NDCG@12':>9}{'MAP@12':>8}")
    for name, by_slice in report["full_ranking"].items():
        m = by_slice["overall"]
        print(f"{name:<28}{m['HitRate@12']:>8.4f}{m['HitRate@50']:>8.4f}{m['HitRate@100']:>9.4f}{m['Recall@100']:>9.4f}{m['NDCG@12']:>9.4f}{m['MAP@12']:>8.4f}")
    for slice_name in ("warm_items", "strict_cold_items", "never_sold_items", "repeat_items", "explore_items"):
        m = report["full_ranking"]["reranker"][slice_name]
        print(f"  reranker/{slice_name:<19}{m['HitRate@12']:>8.4f}{m['HitRate@50']:>8.4f}{m['HitRate@100']:>9.4f}{m['Recall@100']:>9.4f}{m['NDCG@12']:>9.4f}{m['MAP@12']:>8.4f}  (users={m['evaluated_users']})")
    for name, by_slice in report["full_ranking"].items():
        if "cold_slot" in name:
            for slice_name in ("overall", "strict_cold_items", "never_sold_items"):
                m = by_slice[slice_name]
                print(f"  {name}/{slice_name:<18}{m['HitRate@12']:>8.4f}{m['HitRate@50']:>8.4f}{m['HitRate@100']:>9.4f}{m['Recall@100']:>9.4f}{m['NDCG@12']:>9.4f}{m['MAP@12']:>8.4f}  (users={m['evaluated_users']})")
    print("\npaired diff vs recent_popularity (mean, 95% CI):")
    for key, d in report["paired_vs_recent_popularity"].items():
        print(f"  {key:<32}{d['diff']:+.4f}  [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}]")
    if "paired_v2_vs_v1_features" in report:
        for key, d in report["paired_v2_vs_v1_features"].items():
            print(f"  v2 - v1 features {key:<14}{d['diff']:+.4f}  [{d['ci95'][0]:+.4f}, {d['ci95'][1]:+.4f}]")
    print(f"\n{'sampled 1+%d' % SAMPLED_NEG:<46}{'HR@10':>8}{'±ci':>7}{'HR@20':>8}{'NDCG@10':>9}{'MRR':>7}{'pairs':>7}")
    for key, by_scorer in report["sampled_1plus99"].items():
        for name, m in by_scorer.items():
            print(f"{key + ' / ' + name:<46}{m['HitRate@10']:>8.4f}{m['HitRate@10_ci95']:>7.3f}{m['HitRate@20']:>8.4f}{m['NDCG@10']:>9.4f}{m['MRR']:>7.3f}{m['n_pairs']:>7}")

result = {"config": {"serving_rule": SERVING, "item_stats": STATS_SOURCE, "ranker": RANKER, "features": FEATURE_SET, "tower_for_test": TOWER_FOR_TEST if towers["refit"] is not towers["base"] else "base",
                     "ranker_refit": RANKER_REFIT, "rounds": best_rounds, "sampled_neg": SAMPLED_NEG, "cutoff_days": CUTOFF},
          "valid": evaluate_window(valid_groups, valid_sampled, rankers_selection, towers["base"], "valid", valid_context),
          "test": evaluate_window(test_groups, test_sampled, rankers_test, test_tower, "test", test_context)}
print_report("VALID (ranker train trên selection)", result["valid"])
print_report("TEST (tower refit + ranker refit) <- số báo cáo", result["test"])

(OUTPUT_DIR / "reranker_metrics.json").write_text(json.dumps(result, indent=2, default=float))
if RANKER == "lgbm":
    rankers_test["reranker"].model.save_model(str(OUTPUT_DIR / "reranker_lgbm.txt"))
else:
    torch.save({"state": rankers_test["reranker"].model.state_dict(), "features": FEATURES}, OUTPUT_DIR / "best_reranker.pt")

## 7. Xuất model

Gói mọi thứ cần để dùng/đem đi vào `hm_export/` và `hm_export.zip` (tải ở tab Output của Kaggle): tower (đã bỏ hai buffer embedding Jina 2×105k×512 vì nạp lại được từ `.npy`), reranker và `model_meta.json` (thứ tự đặc trưng, luật phục vụ, hằng số candidate, cutoff). Reranker một mình **không đủ** để suy luận: nó cần đúng pipeline đặc trưng ở mục 4 và `item_daily_counts.parquet`.

In [ ]:
import shutil
EXPORT_DIR = Path(os.getenv("HM_EXPORT_DIR", str(OUTPUT_DIR.parent / "hm_export")))
shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True)

tower_src = refit_path if (TOWER_FOR_TEST == "refit" and refit_path.is_file()) else RETRIEVAL_DIR / "best_retrieval.pt"
tower_ckpt = torch.load(tower_src, map_location="cpu", weights_only=False)
tower_ckpt["state"] = {k: v for k, v in tower_ckpt["state"].items() if k not in ("text", "image")}
torch.save(tower_ckpt, EXPORT_DIR / "tower.pt")

ranker_files = {}
for name, scorer in rankers_test.items():
    if scorer.kind == "lgbm":
        scorer.model.save_model(str(EXPORT_DIR / f"{name}_lgbm.txt")); ranker_files[name] = f"{name}_lgbm.txt"
    else:
        torch.save({"state": scorer.model.state_dict(), "mean": scorer.mean, "std": scorer.std, "columns": scorer.columns}, EXPORT_DIR / f"{name}_mlp.pt"); ranker_files[name] = f"{name}_mlp.pt"

meta = {"tower": {"file": "tower.pt", "source": tower_src.name, "modality": MODALITY, "d_model": D_MODEL, "maxlen": MAXLEN, "serving": SERVING, "note": "thiếu buffer text/image: nạp text_embeddings.npy/image_embeddings.npy (thêm hàng 0 đầu) vào Tower.text/Tower.image"},
        "ranker": {"kind": RANKER, "files": ranker_files, "feature_columns": {n: [FEATURES_V2[c] for c in cols] for n, cols in FEATURE_VARIANTS.items()}, "all_features_in_order": FEATURES_V2},
        "candidates": {"sources": SOURCES, "tower_k": TOWER_K, "recent_k": RECENT_K, "repeat_k": REPEAT_K, "new_k": NEW_K, "content_k": CONTENT_K, "popular_k": POPULAR_K, "cold_k": COLD_K, "prior_days": PRIOR_DAYS},
        "cold_slot_every": COLD_EVERY, "item_stats": STATS_SOURCE, "cutoff_days": CUTOFF, "test_metrics_file": "reranker_metrics.json"}
(EXPORT_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2, default=float))
shutil.copy(OUTPUT_DIR / "reranker_metrics.json", EXPORT_DIR / "reranker_metrics.json")
archive = shutil.make_archive(str(EXPORT_DIR), "zip", EXPORT_DIR)
for f in sorted(EXPORT_DIR.iterdir()):
    print(f"{f.name:<28}{f.stat().st_size / 1e6:>9.1f} MB")
print("zip:", archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")

**Báo cáo:** `candidate` đo khả năng giữ ground truth trong union (cận trên của reranker); `full_ranking` xếp hạng toàn catalog (item ngoài union coi như không gợi ý); `sampled_1plus99` chỉ là bảng phụ để so với giao thức Amazon — không so với full-ranking và luôn đọc cạnh `random`/`popularity`/`recent_popularity`.